# Cipher Code Kraken - Stage 2: SimPO Anti-Slop Training

**Purpose:** Preference optimization using SimPO (Simple Preference Optimization) to teach
the model to actively prefer hand-crafted, Awwwards-worthy creative code over generic
AI-generated template garbage. This is the anti-slop stage.

**Input model:** `cipher-sft-merged/` (output from Stage 1 SFT)

**Training data:** 1000+ chosen/rejected pairs where:
- **Chosen:** Three.js scenes, GSAP ScrollTrigger animations, Lenis smooth scroll, custom shaders
- **Rejected:** Tailwind gradient heroes, div soup, animate-bounce, template layouts

**Why SimPO over DPO:**
- Reference-free (no ref model = less VRAM on A100)
- +6.4pts over DPO on AlpacaEval 2
- Uses CPOTrainer from TRL with `loss_type='simpo'`

**Expected runtime:** ~2 hours on A100

**Output:** Merged SimPO adapter saved to `cipher-simpo-merged/` for Stage 3 (GRPO).

**W&B project:** cipher-code-kraken / simpo-stage

## Cell 1: Environment Setup

Install Unsloth and W&B. If continuing from the same Colab session as SFT,
these may already be installed -- the pip install will just verify.

**Expected output:** Successful install, W&B login.

In [ ]:
# Install Unsloth (Colab-optimized) and experiment tracking
!pip install unsloth
!pip install wandb

import wandb
wandb.login()

## Cell 2: Mount Data and Load SFT-Merged Model from Drive

Mount Google Drive and copy the SFT-merged model (from Stage 1) and SimPO training data.

**If running in the same Colab session as SFT:** The `cipher-sft-merged/` directory
may already exist locally. Skip the Drive copy and just verify the directory exists.

**If starting a new Colab session:** Drive mount is required to restore the SFT model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy SFT-merged model from Drive (saved in previous stage)
import os
if not os.path.exists('./cipher-sft-merged'):
    !cp -r /content/drive/MyDrive/cipher-models/cipher-sft-merged ./
    print("SFT-merged model restored from Drive")
else:
    print("SFT-merged model already exists locally")

# Create data directory and copy SimPO training data
!mkdir -p ./data/prompts
!cp /content/drive/MyDrive/cipher-data/simpo_prompts.jsonl ./data/prompts/

# Copy configs
!mkdir -p ./configs
!cp /content/drive/MyDrive/cipher-data/simpo_config.py ./configs/

# Verify
!ls -la ./cipher-sft-merged/ | head -5
!wc -l ./data/prompts/simpo_prompts.jsonl

## Cell 3: Load SFT-Merged Model

Load the SFT-merged model (NOT the base model). This model already knows creative code
patterns from Stage 1. SimPO will now teach it to PREFER creative code over generic templates.

**Key difference from SFT notebook:** `MODEL_ID` points to `./cipher-sft-merged`
instead of the base `unsloth/gemma-4-31B-it`.

**Watch for:** VRAM should be similar to SFT (~18-22GB after load).

In [ ]:
import torch
from unsloth import FastLanguageModel
import sys
sys.path.append('/content')
from configs.simpo_config import *

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,  # cipher-sft-merged
    load_in_4bit=LOAD_IN_4BIT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
)

print(f"SFT-merged model loaded from: {MODEL_ID}")
print(f"VRAM after load: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 4: Apply Fresh LoRA for SimPO

Apply a fresh LoRA adapter on top of the SFT-merged base. This is the standard
adapter-merging pattern (Research Pattern 4): each stage gets its own LoRA adapter
on top of the previously merged model.

**Same LoRA config as SFT:** r=16, alpha=16, gradient checkpointing enabled.
Consistency across stages prevents unexpected behavior.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=RANDOM_STATE,
)

model.print_trainable_parameters()
print(f"VRAM after LoRA: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 5: Load Preference Dataset

Load the SimPO preference pairs. Each entry has:
- `prompt` -- The user request (e.g., "Build a hero section with 3D text")
- `chosen` -- Hand-crafted creative code (Three.js + GSAP + custom shaders)
- `rejected` -- Generic AI slop (Tailwind gradient hero, div soup, animate-bounce)

**CRITICAL (Pitfall 4):** Rejected examples should be competent-but-generic, NOT broken
code. The model needs to learn taste (creative vs generic), not just "working vs broken."

**Sanity checks:**
- Chosen != Rejected (they must actually differ)
- Chosen should be longer (creative code is more substantial)
- 1000+ pairs minimum for meaningful preference learning

In [ ]:
from datasets import load_dataset

# Format: {"prompt": "...", "chosen": "...", "rejected": "..."}
preference_dataset = load_dataset("json", data_files=SIMPO_DATA_PATH)

print(f"Preference pairs: {len(preference_dataset['train'])}")

# Sanity checks
sample = preference_dataset['train'][0]
print(f"\nSample prompt: {sample['prompt'][:100]}...")
print(f"Chosen length: {len(sample['chosen'])} chars")
print(f"Rejected length: {len(sample['rejected'])} chars")
assert sample['chosen'] != sample['rejected'], "Chosen and rejected must differ!"

# Check quality distribution
chosen_lengths = [len(ex['chosen']) for ex in preference_dataset['train']]
rejected_lengths = [len(ex['rejected']) for ex in preference_dataset['train']]
print(f"\nAvg chosen length: {sum(chosen_lengths)/len(chosen_lengths):.0f} chars")
print(f"Avg rejected length: {sum(rejected_lengths)/len(rejected_lengths):.0f} chars")
print(f"Chosen > Rejected in {sum(c > r for c, r in zip(chosen_lengths, rejected_lengths))}/{len(chosen_lengths)} pairs")

## Cell 6: Configure and Run SimPO Training

Train with CPOTrainer from TRL using `loss_type='simpo'` for reference-free preference
optimization.

**SimPO-specific settings (from the paper):**
- `loss_type='simpo'` -- Reference-free (no ref model = less VRAM)
- `cpo_alpha=0.0` -- Pure SimPO, no CPO regularization
- `simpo_gamma=1.4` -- Target reward margin between chosen/rejected
- `beta=2.0` -- SimPO paper default

**Key differences from SFT:**
- `lr=5e-5` (10x lower) -- Preference learning needs gentle updates
- `epochs=1` -- One pass through preference pairs is sufficient
- `max_prompt_length=512` -- Prompts are shorter than full training examples

**Expected behavior:**
- Loss should decrease steadily but NOT drop to near-zero (see Pitfall 4 warning below)
- If loss drops >90% in first 10% of training, rejected data is too easy
- VRAM should be similar to SFT (~32-38GB peak)

**Runtime:** ~2 hours on A100 for 1000 preference pairs

In [ ]:
from trl import CPOTrainer, CPOConfig
import os

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

simpo_config = CPOConfig(
    output_dir=OUTPUT_DIR,
    loss_type=LOSS_TYPE,           # "simpo" -- reference-free
    cpo_alpha=CPO_ALPHA,           # 0.0 = pure SimPO
    simpo_gamma=SIMPO_GAMMA,       # 1.4 reward margin
    beta=BETA,                     # 2.0 per paper
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=BF16,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    report_to=REPORT_TO,
    run_name=WANDB_RUN_NAME,
)

trainer = CPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=preference_dataset["train"],
    args=simpo_config,
)

print(f"Pre-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
trainer.train()
print(f"Post-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
print("SimPO training complete!")

## Cell 7: Monitor Training Quality (Pitfall 4 Check)

**CRITICAL CHECK:** If SimPO loss drops too quickly (>90% decrease), it means the
rejected examples are too easy to distinguish from chosen. The model learns nothing
useful -- it needs hard contrasts (good-but-generic vs genuinely creative).

This cell pulls the loss curve from W&B and checks the early-vs-late loss ratio.

**Healthy:** Loss ratio (late/early) between 0.3-0.8
**Warning:** Loss ratio < 0.1 = rejected data too easy, regenerate with higher quality baseline
**Also warning:** Loss ratio > 0.9 = model not learning preferences, check data format

In [ ]:
# Check that SimPO loss didn't drop to near-zero too quickly
# (Pitfall 4: if loss drops fast, rejected data is too weak)
import wandb

api = wandb.Api()
runs = api.runs(WANDB_PROJECT, filters={"display_name": WANDB_RUN_NAME})

if runs:
    run = runs[0]
    history = run.history()
    
    if len(history) > 10 and 'loss' in history.columns:
        early_loss = history['loss'].dropna().iloc[:10].mean()
        late_loss = history['loss'].dropna().iloc[-10:].mean()
        ratio = late_loss / early_loss if early_loss > 0 else 0
        
        print(f"Early loss (first 10 steps): {early_loss:.4f}")
        print(f"Late loss (last 10 steps): {late_loss:.4f}")
        print(f"Loss ratio (late/early): {ratio:.2f}")
        print()
        
        if ratio < 0.1:
            print("WARNING: Loss dropped >90% -- rejected examples may be too easy!")
            print("Consider regenerating rejected data with higher quality baseline.")
            print("See Research Pitfall 4 for details.")
        elif ratio > 0.9:
            print("WARNING: Loss barely decreased -- model may not be learning preferences.")
            print("Check dataset format and verify chosen/rejected actually differ.")
        else:
            print("Loss ratio looks healthy -- model is learning preferences.")
    else:
        print("Not enough history data to analyze. Check W&B dashboard manually.")
else:
    print("W&B run not found. Check WANDB_PROJECT and WANDB_RUN_NAME settings.")
    print(f"Looking for: {WANDB_PROJECT}/{WANDB_RUN_NAME}")

## Cell 8: Merge SimPO Adapter and Save

Merge the SimPO LoRA adapter into the SFT-merged base to produce the final
`cipher-simpo-merged` model. This model:
1. Knows creative code patterns (from SFT)
2. Actively prefers creative code over generic templates (from SimPO)

**Output:** `cipher-simpo-merged/` -- input for Stage 3 (GRPO reward optimization).

**Backup:** Saved to Google Drive for Colab session persistence.

In [ ]:
# Merge SimPO adapter and save
model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer)
print(f"SimPO-merged model saved to {MERGED_OUTPUT_DIR}")

# Backup to Drive
!mkdir -p /content/drive/MyDrive/cipher-models/
!cp -r {MERGED_OUTPUT_DIR} /content/drive/MyDrive/cipher-models/
print("Backup saved to Google Drive")

## Cell 9: Comparative Eval - SFT vs SFT+SimPO

Generate a response to a creative code prompt and run the slop detector on it.
This tests whether SimPO successfully shifted the model's preferences toward
creative code and away from generic templates.

**What to look for:**
- Response should use Three.js/GSAP/ScrollTrigger (not just CSS animations)
- Should NOT contain: `bg-gradient-to-r`, `animate-bounce`, `hero-section`
- Slop score should be LOW (creative code scores low on slop detection)
- Code should be substantial (100+ lines) with real 3D setup

**Compare with SFT-only output:** If you saved the SFT Stage 1 generation test output,
compare side-by-side. SimPO output should be noticeably more creative and specific.

In [ ]:
# Compare SFT-only vs SFT+SimPO on the same prompt
FastLanguageModel.for_inference(model)

test_prompt = "Build a hero section with 3D text that rotates on scroll using Three.js and GSAP ScrollTrigger"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 80)
print("GENERATION TEST - SFT+SimPO Stage")
print("=" * 80)
print(response)
print("=" * 80)

# Run slop detector on output
try:
    sys.path.append('/content')
    from scripts.slop_detector import slop_score
    score = slop_score(response)
    print(f"\nSlop score: {score['score']:.1f} (is_slop: {score['is_slop']})")
    print(f"Signals: {score['signals']}")
except ImportError:
    print("\nSlop detector not found. Copy scripts/slop_detector.py to Colab to enable.")
    print("Manual check: Does the output use Three.js/GSAP, or is it generic HTML?")

print(f"\nResponse length: {len(response)} chars")
print(f"Final VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")